# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [10]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}


## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [11]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [12]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


In [14]:
(movies == 0).sum()

movieId                    0
tmdbId                     0
imdb_id                    0
title                      0
original_title             0
original_language          0
release_date               0
runtime                   20
budget                  4443
revenue                 4322
popularity                 0
vote_average              48
vote_count                44
genres                     0
production_companies       0
production_countries       0
adult                   9082
status                     0
genres_list                0
primary_genre              0
release_year               0
dtype: int64

### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [13]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:

 What features in a movies will predict user ratings the most? (runtime, genre, cast ex..)

- Why this question matters:

 The answer of this question can suggest what kind of movie, how long to make the movie, and who to cast for the movie, to ensure a higher user rating.

- Tables/files used:

The notebook contains 5 tables that each contain unique information titled. "movies,". "ratings", "links", "credits", and "keywords".

"movies" has general movie metadata such as (title, release date, runtime, genre, budget, and revenue.) 9,082 rows x 21 columns
"ratings" has user ratings. 99,810 rows x 5 columns
"links" has movie ids linked to what looks like some labeling system. 9,125 rows x 3 columns
"credits" has cast and crew information. 9,082 rows x 3 columns
"keywords" contains keywords/tags for movies.  9,082 rows x 2 columns

- Row meaning of each important table:

movies rows: One row is one unique movie.
ratings rows: One rating by a user for a specific movie at a specific time.
links rows: Each movie id is mapped to two other columns .
credits rows: One row is one movies cast and crew list.
keywords rows: One row looks to be one movies list of keywords/tags.

- Early data quality issues:

1. There are standard missing values concentrated in metadata tables keywords:761 missing cells, credits: 114 missing cells, movies: 72 missing cells, links: 13 missing cells)

2. Numerical columns in the movies table contain 0s (budget, revenue, and runtime), suggesting unreported or missing data.

3. The release_date column in movies is stored as a string rather than a date

4. There are significant missing values as seen in the missing_rate dataset with the highest rate at 8.38% coming from the keywords_list table and the lowest with 0 in the ratings table.

- Initial cleaning plan:

For missing values in the categorical type, I will replace the NaN with "unknown" so that it is still a text and it can be filtered out later if need be. The 0 values in the runtime could be imputed with median values, while the 0 dollar budget and revenue entries can be flagged or filtered out if included as features for my business question. To prevent extreme average rating values caused by low vote counts (like a single one star review). I should probably implement a minimum needed rating number threshhold. (ex 10 minimum ratings). Wrong data types like the release_date will be properly converted to datetime format so that it can properly be used.


- Possible target/outcome:

The target outcome is to try and run an occupancy type of model that can predict what variables (or features) will affect the user rating in a positive or negative way.

I did something similar to this as independant research when i was an undergrad running occupancy models to see what variables (water velocity, temperature, substrate type ect...) affected invasive catfish occupancy in a spring. It seems like I might be able to do something similar with this dataset.



- Ethics, privacy, or governance concerns:

The ratings data includes user ID's and timestamps. User privacy needs to be protected.




In [ ]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.

# checking which colums in movies contain 0
movies.isna().sum()[movies.isna().sum() > 0]

suspicious_zeros = pd.Series({
    'zero_budget_count': (movies['budget'] == 0).sum(),
    'zero_revenue_count': (movies['revenue'] == 0).sum(),
    'zero_runtime_count': (movies['runtime'] == 0).sum()
})
print(suspicious_zeros)


# stats for user ratings
print("Rating Statistics:")
print(ratings['rating'].describe())










zero_budget_count     4443
zero_revenue_count    4322
zero_runtime_count      20
dtype: int64
Rating Statistics:
count    99810.000000
mean         3.542972
std          1.057939
min          0.500000
25%          3.000000
50%          4.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64


In [15]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


...

,dtype
tmdbId,int64
director,str
top_cast,str


,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [ ]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



In [ ]:
rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


In [ ]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [ ]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("movies_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT movieId) AS unique_movie_ids,
            SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
        FROM movies_abt;
        """,
        connection,
    )

sql_abt_check


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [ ]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [ ]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [ ]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
